# 08. Retention Recommendations

This notebook turns churn-risk predictions and customer segments into practical retention campaign recommendations. The goal is to move from model output to business action: who should be contacted, why they are prioritized, what message should be used, and which KPI should be monitored.

**Primary inputs**

- `data/processed/customer_modeling_90d.csv`
- `data/processed/customer_segments.csv`

**Primary outputs**

- `data/processed/customer_retention_recommendations.csv`
- `data/processed/retention_action_summary.csv`
- `data/processed/top_retention_targets.csv`

**Rerun safety**

`SAVE_OUTPUTS` is set to `False` by default. Normal reruns will not overwrite saved recommendation files. If you intentionally set `SAVE_OUTPUTS = True`, the save helper still compares generated CSV content to the existing file and skips the write when nothing changed.

## 1. Safety Controls

In [ ]:
SAVE_OUTPUTS = False
RANDOM_STATE = 42
TEST_SIZE = 0.25
SELECTED_THRESHOLD = 0.40
HIGH_RISK_THRESHOLD = 0.75
TOP_TARGET_COUNT = 100

print("SAVE_OUTPUTS:", SAVE_OUTPUTS)
print("Selected threshold:", SELECTED_THRESHOLD)
print("High-risk threshold:", HIGH_RISK_THRESHOLD)

## 2. Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
    sns.set_theme(style="whitegrid", context="notebook")
except ImportError:
    HAS_PLOTS = False
    print("Plotting libraries are not available. Chart cells will be skipped.")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

## 3. Project Paths

In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else Path(
    r"C:/Learning/BANA8083/AI-retention-decision-support"
)

processed_path = PROJECT_ROOT / "data" / "processed"

CUSTOMER_MODEL_PATH = processed_path / "customer_modeling_90d.csv"
CUSTOMER_SEGMENTS_PATH = processed_path / "customer_segments.csv"
RECOMMENDATIONS_PATH = processed_path / "customer_retention_recommendations.csv"
ACTION_SUMMARY_PATH = processed_path / "retention_action_summary.csv"
TOP_TARGETS_PATH = processed_path / "top_retention_targets.csv"

paths = pd.DataFrame(
    {
        "asset": [
            "customer modeling data",
            "customer segments",
            "retention recommendations output",
            "retention action summary output",
            "top retention targets output",
        ],
        "path": [
            CUSTOMER_MODEL_PATH,
            CUSTOMER_SEGMENTS_PATH,
            RECOMMENDATIONS_PATH,
            ACTION_SUMMARY_PATH,
            TOP_TARGETS_PATH,
        ],
        "exists": [
            CUSTOMER_MODEL_PATH.exists(),
            CUSTOMER_SEGMENTS_PATH.exists(),
            RECOMMENDATIONS_PATH.exists(),
            ACTION_SUMMARY_PATH.exists(),
            TOP_TARGETS_PATH.exists(),
        ],
    }
)
paths

## 4. Load Modeling and Segmentation Data

In [ ]:
for required_path in [CUSTOMER_MODEL_PATH, CUSTOMER_SEGMENTS_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required input: {required_path}")

customer_model = pd.read_csv(
    CUSTOMER_MODEL_PATH,
    parse_dates=["first_purchase_date", "last_purchase_date"],
)
customer_segments = pd.read_csv(CUSTOMER_SEGMENTS_PATH)

print("Customer model:", customer_model.shape)
print("Customer segments:", customer_segments.shape)
customer_model.head()

In [ ]:
required_model_columns = [
    "CustomerID",
    "first_purchase_date",
    "last_purchase_date",
    "frequency",
    "monetary_value",
    "recency",
    "product_diversity",
    "primary_country",
    "active_90d",
    "inactive_90d",
]
required_segment_columns = [
    "CustomerID",
    "customer_segment",
    "R_score",
    "F_score",
    "M_score",
    "RFM_score",
]

missing_model_columns = [col for col in required_model_columns if col not in customer_model.columns]
missing_segment_columns = [col for col in required_segment_columns if col not in customer_segments.columns]

if missing_model_columns:
    raise ValueError(f"Missing model columns: {missing_model_columns}")
if missing_segment_columns:
    raise ValueError(f"Missing segment columns: {missing_segment_columns}")

data_quality = pd.DataFrame(
    {
        "check": [
            "model_rows",
            "model_unique_customers",
            "model_duplicate_customers",
            "segment_rows",
            "segment_unique_customers",
            "segment_duplicate_customers",
            "inactive_rate",
        ],
        "value": [
            len(customer_model),
            customer_model["CustomerID"].nunique(),
            customer_model["CustomerID"].duplicated().sum(),
            len(customer_segments),
            customer_segments["CustomerID"].nunique(),
            customer_segments["CustomerID"].duplicated().sum(),
            customer_model["inactive_90d"].mean(),
        ],
    }
)
data_quality

## 5. Prepare Features and Target

In [ ]:
target = "inactive_90d"

drop_cols = [
    "CustomerID",
    "first_purchase_date",
    "last_purchase_date",
    "active_90d",
    "inactive_90d",
]

X = customer_model.drop(columns=drop_cols)
y = customer_model[target]

categorical_cols = ["primary_country"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "test", "full scoring base"],
        "rows": [len(X_train), len(X_test), len(X)],
        "inactive_rate": [y_train.mean(), y_test.mean(), y.mean()],
    }
)
split_summary

## 6. Refit Final Model for Recommendation Scoring

In [ ]:
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            GradientBoostingClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

final_model.fit(X_train, y_train)

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= SELECTED_THRESHOLD).astype(int)

print(f"Test ROC-AUC: {roc_auc_score(y_test, test_proba):.4f}")
print("Confusion matrix at selected threshold:")
print(confusion_matrix(y_test, test_pred))
print("Classification report at selected threshold:")
print(classification_report(y_test, test_pred, zero_division=0))

In [ ]:
scoring_model = final_model.fit(X, y)

scored_customers = customer_model.copy()
scored_customers["predicted_inactive_probability"] = scoring_model.predict_proba(X)[:, 1]
scored_customers["predicted_inactive_90d"] = (
    scored_customers["predicted_inactive_probability"] >= SELECTED_THRESHOLD
).astype(int)

scored_customers[
    ["CustomerID", "predicted_inactive_probability", "predicted_inactive_90d"]
].head()

## 7. Merge Risk Scores With Segments

In [ ]:
segment_cols = [
    "CustomerID",
    "customer_segment",
    "R_score",
    "F_score",
    "M_score",
    "RFM_score",
]

recommendations = scored_customers.merge(
    customer_segments[segment_cols],
    on="CustomerID",
    how="left",
    validate="one_to_one",
)

missing_segments = recommendations["customer_segment"].isna().sum()
print(f"Rows after merge: {len(recommendations):,}")
print(f"Customers missing segment assignment: {missing_segments:,}")

recommendations.head()

## 8. Create Risk Tiers

In [ ]:
def assign_risk_tier(probability):
    if probability >= HIGH_RISK_THRESHOLD:
        return "High Risk"
    if probability >= SELECTED_THRESHOLD:
        return "Medium Risk"
    return "Low Risk"


recommendations["risk_tier"] = recommendations["predicted_inactive_probability"].apply(assign_risk_tier)

risk_order = ["High Risk", "Medium Risk", "Low Risk"]
recommendations["risk_tier"] = pd.Categorical(
    recommendations["risk_tier"],
    categories=risk_order,
    ordered=True,
)

risk_summary = (
    recommendations["risk_tier"]
    .value_counts(sort=False)
    .rename_axis("risk_tier")
    .reset_index(name="customers")
)
risk_summary["customer_percentage"] = risk_summary["customers"].div(len(recommendations))
risk_summary

## 9. Marketing Strategy Rules

In [ ]:
def default_strategy():
    return {
        "recommended_action": "Monitor / no immediate action",
        "offer_level": "No offer",
        "campaign_objective": "Monitor engagement",
        "message_angle": "Keep customer warm without over-discounting",
        "subject_line": "New picks are waiting for you",
        "preheader": "Explore recommendations based on what customers are loving now.",
        "cta": "Explore New Arrivals",
        "cadence": "No immediate campaign; include in regular newsletter",
        "primary_kpi": "Email engagement rate",
    }


def recommend_marketing_strategy(row):
    segment = row["customer_segment"]
    risk_tier = row["risk_tier"]

    strategy = default_strategy()

    if segment == "High-Value At-Risk Customers" and risk_tier == "High Risk":
        strategy = {
            "recommended_action": "VIP win-back campaign",
            "offer_level": "Premium offer",
            "campaign_objective": "Recover high-value customers before they become fully dormant",
            "message_angle": "Exclusive, personal, high-touch reactivation",
            "subject_line": "A special offer, just for you",
            "preheader": "We saved something extra for valued customers like you.",
            "cta": "Redeem Your Exclusive Offer",
            "cadence": "Email 1 now, reminder in 5 days, final reminder in 10 days",
            "primary_kpi": "Reactivation revenue",
        }
    elif segment == "High-Value At-Risk Customers" and risk_tier == "Medium Risk":
        strategy = {
            "recommended_action": "Personalized win-back offer",
            "offer_level": "Moderate offer",
            "campaign_objective": "Re-engage valuable customers with personalized product recommendations",
            "message_angle": "Value reminder + curated recommendations",
            "subject_line": "We picked these with your past favorites in mind",
            "preheader": "Come back to products that match your shopping history.",
            "cta": "View Your Picks",
            "cadence": "Email now, product reminder in 7 days",
            "primary_kpi": "Repeat purchase rate",
        }
    elif segment == "Dormant Customers" and risk_tier == "High Risk":
        strategy = {
            "recommended_action": "Low-cost reactivation campaign",
            "offer_level": "Low-cost offer",
            "campaign_objective": "Reactivate dormant customers while controlling promotion cost",
            "message_angle": "What is new since your last visit",
            "subject_line": "A lot has changed since your last order",
            "preheader": "See new arrivals and limited-time picks you may have missed.",
            "cta": "See What is New",
            "cadence": "Email now, final reminder in 7 days",
            "primary_kpi": "Reactivation rate",
        }
    elif segment == "Dormant Customers" and risk_tier == "Medium Risk":
        strategy = {
            "recommended_action": "Browse-based reactivation",
            "offer_level": "Low-cost offer",
            "campaign_objective": "Encourage browsing before offering deeper incentives",
            "message_angle": "Discovery and newness",
            "subject_line": "Still interested? Here is what is new",
            "preheader": "Fresh products and customer favorites are waiting.",
            "cta": "Browse New Arrivals",
            "cadence": "Email now; suppress if no engagement after 14 days",
            "primary_kpi": "Click-through rate",
        }
    elif segment == "Low-Value Occasional Customers" and risk_tier in ["High Risk", "Medium Risk"]:
        strategy = {
            "recommended_action": "Automated bundle or value campaign",
            "offer_level": "Low-cost offer",
            "campaign_objective": "Increase repeat purchase without using expensive incentives",
            "message_angle": "Affordable bundles and easy add-ons",
            "subject_line": "Small finds you may love",
            "preheader": "Affordable picks based on your past shopping behavior.",
            "cta": "Shop Value Picks",
            "cadence": "Automated email now, bundle reminder in 7 days",
            "primary_kpi": "Conversion rate",
        }
    elif segment == "New Customers" and risk_tier in ["High Risk", "Medium Risk"]:
        strategy = {
            "recommended_action": "Second-purchase nurture campaign",
            "offer_level": "Welcome / onboarding offer",
            "campaign_objective": "Convert first-time buyers into repeat customers",
            "message_angle": "Welcome journey + next best purchase",
            "subject_line": "Ready for your next favorite find?",
            "preheader": "Here are a few picks to continue your shopping experience.",
            "cta": "Shop Your Recommendations",
            "cadence": "Email now, follow-up in 5 days, educational content in 10 days",
            "primary_kpi": "Second purchase rate",
        }
    elif segment == "Regular Customers" and risk_tier in ["High Risk", "Medium Risk"]:
        strategy = {
            "recommended_action": "Personalized replenishment / cross-sell reminder",
            "offer_level": "Standard offer",
            "campaign_objective": "Maintain repeat purchase behavior",
            "message_angle": "Helpful reminder based on prior purchases",
            "subject_line": "You may be due for a restock",
            "preheader": "We found a few items that pair well with your past orders.",
            "cta": "Shop Recommended Items",
            "cadence": "Email now, cross-sell reminder in 7 days",
            "primary_kpi": "Repeat purchase rate",
        }
    elif segment == "Loyal Customers" and risk_tier in ["High Risk", "Medium Risk"]:
        strategy = {
            "recommended_action": "Loyalty protection campaign",
            "offer_level": "Relationship-building offer",
            "campaign_objective": "Protect loyal customer relationship and prevent disengagement",
            "message_angle": "Appreciation, exclusivity, and early access",
            "subject_line": "A thank-you for being one of our best customers",
            "preheader": "Enjoy early access and loyalty benefits before everyone else.",
            "cta": "Unlock Your Loyalty Benefit",
            "cadence": "Email now, loyalty reminder in 10 days",
            "primary_kpi": "Loyal customer retention rate",
        }

    return pd.Series(strategy)


strategy_cols = recommendations.apply(recommend_marketing_strategy, axis=1)
recommendations = pd.concat([recommendations, strategy_cols], axis=1)

recommendations[
    [
        "CustomerID",
        "customer_segment",
        "risk_tier",
        "predicted_inactive_probability",
        "monetary_value",
        "recommended_action",
        "offer_level",
        "campaign_objective",
        "subject_line",
        "cta",
    ]
].head()

## 10. Create Message Copy

In [ ]:
def create_marketing_message(row):
    action = row["recommended_action"]
    recency = int(row["recency"])
    frequency = int(row["frequency"])

    if action == "VIP win-back campaign":
        return (
            f"We noticed it has been {recency} days since your last order. "
            "As one of our higher-value customers, we wanted to give you early access to a personalized offer. "
            "Come back and discover picks selected around your past shopping behavior."
        )
    if action == "Personalized win-back offer":
        return (
            "It has been a little while since your last purchase, so we curated a few recommendations based on your past orders. "
            "Your shopping history shows strong engagement, and these picks are designed to make your next order easier."
        )
    if action == "Low-cost reactivation campaign":
        return (
            "A lot has changed since your last visit. Explore new arrivals, customer favorites, "
            "and limited-time picks that may be worth another look."
        )
    if action == "Browse-based reactivation":
        return "Still browsing? We pulled together new products and popular picks to help you rediscover what is available."
    if action == "Automated bundle or value campaign":
        return "Based on your past shopping behavior, we found a few value-friendly bundles and add-ons that may fit your style."
    if action == "Second-purchase nurture campaign":
        return (
            "Thanks for your first order. To help you continue your shopping experience, "
            "we selected a few next-step recommendations that pair well with what customers often buy next."
        )
    if action == "Personalized replenishment / cross-sell reminder":
        return (
            f"You have ordered from us {frequency} times before, so we selected a few items that may pair well with your past purchases. "
            "Take a look before your next restock."
        )
    if action == "Loyalty protection campaign":
        return (
            "Thank you for being a loyal customer. You have built a strong shopping history with us, "
            "so we wanted to share early access and loyalty benefits before everyone else."
        )
    return "No immediate outreach is recommended for this customer. Continue monitoring behavior and include them in standard marketing communication."


recommendations["message_template"] = recommendations.apply(create_marketing_message, axis=1)

recommendations[
    [
        "CustomerID",
        "customer_segment",
        "risk_tier",
        "subject_line",
        "preheader",
        "message_template",
        "cta",
    ]
].head(10)

## 11. Add Campaign Priority Score

In [ ]:
recommendations["campaign_priority_score"] = (
    recommendations["predicted_inactive_probability"] * np.log1p(recommendations["monetary_value"].clip(lower=0))
)

recommendations = recommendations.sort_values(
    "campaign_priority_score",
    ascending=False,
).reset_index(drop=True)

recommendations[
    [
        "CustomerID",
        "customer_segment",
        "risk_tier",
        "predicted_inactive_probability",
        "monetary_value",
        "campaign_priority_score",
        "recommended_action",
        "offer_level",
        "subject_line",
        "cta",
    ]
].head(20)

## 12. Campaign Summary

In [ ]:
action_summary = (
    recommendations.groupby(
        [
            "customer_segment",
            "risk_tier",
            "recommended_action",
            "offer_level",
            "campaign_objective",
            "primary_kpi",
        ],
        observed=False,
    )
    .agg(
        customers=("CustomerID", "count"),
        avg_predicted_inactive_probability=("predicted_inactive_probability", "mean"),
        avg_monetary_value=("monetary_value", "mean"),
        total_monetary_value=("monetary_value", "sum"),
        avg_campaign_priority_score=("campaign_priority_score", "mean"),
    )
    .reset_index()
)

action_summary = action_summary[action_summary["customers"] > 0]
action_summary = action_summary.sort_values(
    ["risk_tier", "avg_campaign_priority_score"],
    ascending=[True, False],
).reset_index(drop=True)

action_summary

In [ ]:
if HAS_PLOTS:
    plot_summary = action_summary.sort_values("customers", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.barplot(data=plot_summary, x="customers", y="recommended_action", hue="risk_tier", ax=ax)
    ax.set_title("Retention Campaign Volume by Recommended Action")
    ax.set_xlabel("Customers")
    ax.set_ylabel("")
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart because plotting libraries are not available.")

## 13. Top Campaign Targets

In [ ]:
top_targets = recommendations[
    recommendations["recommended_action"] != "Monitor / no immediate action"
].copy()

top_targets = top_targets.sort_values(
    "campaign_priority_score",
    ascending=False,
).head(TOP_TARGET_COUNT)

top_targets[
    [
        "CustomerID",
        "customer_segment",
        "risk_tier",
        "predicted_inactive_probability",
        "monetary_value",
        "campaign_priority_score",
        "recommended_action",
        "offer_level",
        "subject_line",
        "preheader",
        "message_template",
        "cta",
        "cadence",
        "primary_kpi",
    ]
].head(20)

## 14. Output Validation

In [ ]:
output_validation = pd.DataFrame(
    {
        "check": [
            "recommendation_rows",
            "unique_customers",
            "missing_segments",
            "missing_recommended_actions",
            "top_target_rows",
            "customers_selected_for_action",
        ],
        "value": [
            len(recommendations),
            recommendations["CustomerID"].nunique(),
            recommendations["customer_segment"].isna().sum(),
            recommendations["recommended_action"].isna().sum(),
            len(top_targets),
            (recommendations["recommended_action"] != "Monitor / no immediate action").sum(),
        ],
    }
)
output_validation

## 15. Save Outputs Safely

In [ ]:
def save_csv_if_changed(dataframe, output_path):
    """Save a CSV only when saving is enabled and the file content changed."""
    if not SAVE_OUTPUTS:
        print(f"SAVE_OUTPUTS=False. Skipped save: {output_path}")
        return "skipped_save_outputs_false"

    processed_path.mkdir(parents=True, exist_ok=True)
    new_csv = dataframe.to_csv(index=False)

    if output_path.exists():
        existing_csv = output_path.read_text(encoding="utf-8")
        if existing_csv == new_csv:
            print(f"No changes detected. Skipped save: {output_path}")
            return "unchanged"

    output_path.write_text(new_csv, encoding="utf-8")
    print(f"Saved updated file: {output_path}")
    return "updated"


save_status = {
    "customer_retention_recommendations": save_csv_if_changed(recommendations, RECOMMENDATIONS_PATH),
    "retention_action_summary": save_csv_if_changed(action_summary, ACTION_SUMMARY_PATH),
    "top_retention_targets": save_csv_if_changed(top_targets, TOP_TARGETS_PATH),
}

save_status

## 16. Recommendation Takeaways

- Retention actions combine model risk, customer value, and segment context.
- High-risk, high-value customers should receive the strongest win-back treatment.
- Dormant and low-value occasional customers should receive lower-cost automated campaigns to control incentive spend.
- Loyal and regular customers should be protected with relationship-building, replenishment, and cross-sell messaging instead of unnecessary discounts.
- The top-target list is designed for campaign execution, while the action summary is designed for reporting and business review.

Keep `SAVE_OUTPUTS = False` for normal reruns. Change it only when you intentionally want to refresh the saved recommendation artifacts.